# 🌐 NanoMultimodal Training & Inference

Complete demonstration of the multimodal AI lab with:
- **CLIP** (vision-language alignment)
- **VAE** (latent compression)
- **Diffusion** (text-to-image generation)
- **Captioning** (image-to-text)
- **Language Model** (text-to-text)
- **🌀 Dual Oscillation** (creative 180° rotation blending)

---

## Setup

Import all necessary modules from nano_moe.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torchvision as tv

from nano_moe.train_multimodal import (
    ExtendedMultimodal,
    FashionCaptionDataset,
    train_joint,
    text_to_image,
    image_to_text,
    image_to_image,
    text_to_text,
    show_grid
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {DEVICE}")

## Configuration

Set training hyperparameters.

In [ ]:
# Training config
USE_LATENT = False  # True for VAE+LDM (slower), False for pixel DDPM (faster)
EPOCHS = 2
BATCH_SIZE = 128
EMB_DIM = 128
STEPS = 200
MAX_LEN = 16
IMG_SIZE = 32 if USE_LATENT else 28

# Loss weights
LAM_CLIP = 1.0
LAM_CAP = 1.0
LAM_DIFF = 1.0
LAM_LM = 1.0

print(f"📊 Config: mode={'LATENT' if USE_LATENT else 'PIXEL'}, epochs={EPOCHS}, batch={BATCH_SIZE}")

## Stage 1: Train VAE (Latent Mode Only)

If using latent diffusion, we first train a VAE to learn compressed representations.

In [ ]:
vae = None

if USE_LATENT:
    print("🎨 Training VAE for latent compression...")
    from nano_moe.train_diffusion import train_vae
    vae = train_vae(epochs=2, batch_size=BATCH_SIZE, img_size=IMG_SIZE)
    print("✅ VAE training complete!")
else:
    print("⏭️ Skipping VAE (using pixel-space DDPM)")

## Stage 2: Joint Multimodal Training

Train a unified model with 4 objectives:
1. **CLIP Contrastive**: Align images and text
2. **Image Captioning**: Generate text from images
3. **Diffusion**: Generate images from text
4. **Language Model**: Generate text from text

In [ ]:
print("🌐 Starting joint multimodal training...")

model, train_ds, val_ds = train_joint(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    emb_dim=EMB_DIM,
    steps=STEPS,
    lam_clip=LAM_CLIP,
    lam_cap=LAM_CAP,
    lam_diff=LAM_DIFF,
    lam_lm=LAM_LM,
    max_len=MAX_LEN,
    use_latent=USE_LATENT,
    vae=vae,
    img_size=IMG_SIZE
)

stoi = train_ds.stoi
itos = train_ds.itos

print("✅ Joint training complete!")

## Inference Demo 1: Text → Image

Generate images from text descriptions using the trained diffusion model.

In [ ]:
print("\n🖼️ Demo: Text → Image")
print("=" * 50)

prompts = [
    "a stylish sneaker",
    "an elegant dress",
    "a clear bag",
    "a bold t-shirt"
]

for prompt in prompts:
    imgs = text_to_image(
        model, 
        prompt, 
        stoi, 
        max_len=MAX_LEN, 
        n=6, 
        guide_w=2.0,
        use_ddim=USE_LATENT,
        ddim_steps=50
    )

## Inference Demo 2: Image → Text

Generate captions for images using the captioning decoder.

In [ ]:
print("\n📝 Demo: Image → Text")
print("=" * 50)

# Sample from validation set
sample_loader = DataLoader(val_ds, batch_size=4, shuffle=True)
sample_imgs, _, true_caps, _, _ = next(iter(sample_loader))

# Show images
show_grid(sample_imgs, "Sample Images")

# Generate captions
generated_caps = image_to_text(model, sample_imgs, itos)

print("\n📊 Results:")
for i, (true, gen) in enumerate(zip(true_caps, generated_caps)):
    print(f"  [{i}] True: '{true}'")
    print(f"      Gen:  '{gen}'")

## Inference Demo 3: Image → Image

Transform images with optional text conditioning (style transfer).

In [ ]:
print("\n🎨 Demo: Image → Image (with text condition)")
print("=" * 50)

# Use one of the sampled images
source_img = sample_imgs[0]

# Show original
show_grid(source_img.unsqueeze(0), "Original")

# Transform with text condition
result = image_to_image(
    model,
    source_img,
    stoi,
    text="a stylish sneaker",
    w_img=0.6,
    w_txt=0.4,
    strength=0.6,
    guide_w=2.5,
    use_ddim=USE_LATENT
)

## Inference Demo 4: Text → Text

Generate text continuations using the decoder-only language model.

In [ ]:
print("\n💬 Demo: Text → Text (Language Model)")
print("=" * 50)

prompts_t2t = [
    "a stylish",
    "an elegant",
    "a clear",
    "a bold"
]

for prompt in prompts_t2t:
    continuation = text_to_text(
        model,
        prompt,
        stoi,
        itos,
        max_len=MAX_LEN,
        max_new_tokens=8,
        temperature=0.9,
        top_k=20,
        top_p=0.95
    )

## 🌀 Special Demo: Dual Oscillation

**The Creative 180° Rotation Feature!**

This generates images that **oscillate between two classes** during denoising,
using periodic 180° rotations to encourage creative blending.

In [ ]:
print("\n🌀 Demo: Dual Oscillation (180° Rotation Blending)")
print("=" * 50)

CLASS_NAMES = [
    "t-shirt", "trouser", "pullover", "dress", "coat",
    "sandal", "shirt", "sneaker", "bag", "ankle boot"
]

# Try different class pairs
oscillation_pairs = [
    (5, 7),  # sandal ↔ sneaker
    (3, 2),  # dress ↔ pullover
    (8, 4),  # bag ↔ coat
]

for class_a, class_b in oscillation_pairs:
    print(f"\n🔄 Oscillating: {CLASS_NAMES[class_a]} ↔ {CLASS_NAMES[class_b]}")
    
    if USE_LATENT:
        # Latent diffusion oscillation
        result = model.diff.dual_oscillation(
            size=(model.vae.latent_channels, 8, 8),
            class_a=class_a,
            class_b=class_b,
            guide_weight=2.0,
            flip_every=50,
            n_classes=10
        )
        result = model.vae.decode(result)
    else:
        # Pixel diffusion oscillation
        result = model.diff.dual_oscillation(
            size=(1, 28, 28),
            class_a=class_a,
            class_b=class_b,
            guide_weight=2.0,
            flip_every=50,
            n_classes=10
        )
    
    show_grid(result, f"Oscillation: {CLASS_NAMES[class_a]} ↔ {CLASS_NAMES[class_b]}")

print("\n✨ The oscillation creates artistic blends between concepts!")

## Summary

🎉 **Congratulations!** You've trained and tested a complete multimodal AI system with:

✅ **Vision-Language Alignment** (CLIP)
✅ **Image Captioning** (Image → Text)
✅ **Text-to-Image Generation** (Text → Image)
✅ **Image-to-Image Transformation** (Style Transfer)
✅ **Language Modeling** (Text → Text)
✅ **Creative Oscillation** (180° Rotation Blending)

---

### Next Steps:

1. **Experiment with hyperparameters**: Try different loss weights, guidance scales, etc.
2. **Extend the dataset**: Add more diverse captions or use different datasets
3. **Fine-tune on custom data**: Adapt the model to your specific use case
4. **Export for deployment**: Save the best checkpoint and deploy!

### Model Capabilities:

| Task | Input | Output | Method |
|------|-------|--------|--------|
| Text→Image | Text description | Generated image | Diffusion (DDPM/LDM) |
| Image→Text | Image | Caption | Transformer Decoder |
| Image→Image | Image + Text | Transformed image | Guided diffusion |
| Text→Text | Text prompt | Continuation | Decoder-only LM |
| Retrieval | Image/Text | Matching text/image | CLIP embeddings |
| Oscillation | Two class IDs | Blended image | 180° rotation + diffusion |

---

**Enjoy your Full AI Lab! 🚀**